# LAB 8

## Task 1: Teoría

### 1. Investigar el algoritmo AC-3 y su relación con el algoritmo de backtracking search

El algoritmo **AC-3 (Arc Consistency Algorithm #3)** es un método utilizado en problemas de satisfacción de restricciones (CSP) para reducir el espacio de búsqueda eliminando valores inconsistentes en las variables antes o durante la búsqueda. Su objetivo principal es lograr consistencia de arco, asegurando que para cada par de variables conectadas por una restricción, los valores asignados a una variable tengan al menos un valor compatible en la otra.

La relación con el algoritmo **Backtracking Search** radica en que AC-3 generalmente se utiliza como un paso previo o complemento a la búsqueda por backtracking. Aplicar AC-3 antes de iniciar la búsqueda ayuda a disminuir considerablemente la cantidad de decisiones incorrectas que el algoritmo de backtracking podría tomar, reduciendo el tamaño del árbol de búsqueda y mejorando significativamente la eficiencia del proceso.

### 2. Defina en sus propias palabras el término “Arc Consistency”

El término **Arc Consistency (Consistencia de Arco)** en un problema CSP se refiere al estado en el cual todas las variables conectadas mediante restricciones han eliminado los valores incompatibles. En otras palabras, una variable es arc-consistente respecto a otra cuando cada uno de sus posibles valores tiene al menos una opción compatible en la variable vecina. Lograr consistencia de arco implica garantizar que ninguna variable tenga valores imposibles o incompatibles, facilitando así encontrar soluciones válidas para el problema.


In [ ]:
import time
import random


# Días disponibles
days = ["Lunes", "Martes", "Miércoles"]

# Inscripción de 4 estudiantes, cada uno con 3 exámenes.
# Algunos exámenes se repiten entre estudiantes, y deben rendirlos en días distintos.
students = {
    "A": ["Exam1", "Exam2", "Exam3"],
    "B": ["Exam1", "Exam4", "Exam5"],
    "C": ["Exam2", "Exam6", "Exam7"],
    "D": ["Exam3", "Exam5", "Exam7"]
}

variables = []
for student, exam_list in students.items():
    for exam in exam_list:
        variables.append((student, exam))


def is_consistent(assignment, var, value):
    """
    Restricciones:
      1) Un estudiante no puede tener dos exámenes en el mismo día.
      2) El mismo examen, si lo tienen varios estudiantes, NO se puede rendir en el mismo día.
    var = (estudiante, examen)
    value = día propuesto
    """
    student, exam = var
    for (s, e), d in assignment.items():

        if s == student and d == value:
            return False
        if e == exam and d == value:
            return False
    return True


# 1. Algoritmo de Backtracking
def backtracking(assignment):
    if len(assignment) == len(variables):
        return assignment  # Asignación completa
    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]
    for value in days:
        if is_consistent(assignment, var, value):
            assignment[var] = value
            result = backtracking(assignment)
            if result is not None:
                return result
            del assignment[var]  # Retroceso
    return None


# 2. Algoritmo de Beam Search
def beam_search(beam_width=50):

    initial_state = {}
    beam = [initial_state]
    while beam:
        new_beam = []
        for state in beam:
            if len(state) == len(variables):
                return state  # Solución completa encontrada
            unassigned = [v for v in variables if v not in state]
            if not unassigned:
                return state
            var = unassigned[0]
            for value in days:
                if is_consistent(state, var, value):
                    new_state = state.copy()
                    new_state[var] = value
                    new_beam.append(new_state)
        if not new_beam:
            return None
        # Ordenar por número de variables asignadas (más asignadas = mejor)
        new_beam.sort(key=lambda s: len(s), reverse=True)
        beam = new_beam[:beam_width]
    return None


# 3. Algoritmo de Local Search 
def count_conflicts(assignment, var, value):

    student, exam = var
    conflicts_count = 0
    for (s, e), d in assignment.items():
        if (s, e) != var:
            # Si es el mismo estudiante y el día es igual, es conflicto
            if s == student and d == value:
                conflicts_count += 1
            # Si es el mismo examen y el día es igual, es conflicto
            if e == exam and d == value:
                conflicts_count += 1
    return conflicts_count

def min_conflicts_single_run(max_iter=5000):

    # Asignación completa aleatoria
    assignment = {var: random.choice(days) for var in variables}
    for _ in range(max_iter):
        conflicted = []
        for var in variables:
            if count_conflicts(assignment, var, assignment[var]) > 0:
                conflicted.append(var)
        if not conflicted:
            return assignment  # Solución sin conflictos encontrada
        var = random.choice(conflicted)
        best_value = None
        best_conflict = float('inf')
        for value in days:
            c = count_conflicts(assignment, var, value)
            if c < best_conflict:
                best_conflict = c
                best_value = value
        assignment[var] = best_value
    return None

def min_conflicts(max_iter=5000, restarts=10):

    for _ in range(restarts):
        sol = min_conflicts_single_run(max_iter=max_iter)
        if sol is not None:
            return sol
    return None


def print_schedule(assignment):
    if assignment is None:
        print("No se encontró solución.\n")
        return
    # Reorganizar la información: para cada estudiante, listar sus exámenes y el día asignado.
    schedule = {}
    for (student, exam), day in assignment.items():
        schedule.setdefault(student, []).append((exam, day))
    for student in sorted(schedule.keys()):
        print(f"Estudiante {student}:")
        for exam, day in schedule[student]:
            print(f"  {exam}: {day}")
        print()


# Backtracking
start = time.time()
solution_bt = backtracking({})
bt_time = time.time() - start
print("=== Backtracking ===")
print_schedule(solution_bt)
print(f"Tiempo: {bt_time:.4f} s\n{'-'*40}\n")

# Beam Search
start = time.time()
solution_beam = beam_search(beam_width=50)
beam_time = time.time() - start
print("=== Beam Search ===")
if solution_beam is None:
    print("No se encontró solución con Beam Search.")
else:
    print_schedule(solution_beam)
print(f"Tiempo: {beam_time:.4f} s\n{'-'*40}\n")

# Local Search 
start = time.time()
solution_local = min_conflicts(max_iter=5000, restarts=10)
local_time = time.time() - start
print("=== Local Search ===")
if solution_local is None:
    print("No se encontró solución con Local Search.")
else:
    print_schedule(solution_local)
print(f"Tiempo: {local_time:.4f} s\n{'-'*40}\n")


=== Backtracking ===
Estudiante A:
  Exam1: Lunes
  Exam2: Martes
  Exam3: Miércoles

Estudiante B:
  Exam1: Martes
  Exam4: Lunes
  Exam5: Miércoles

Estudiante C:
  Exam2: Lunes
  Exam6: Miércoles
  Exam7: Martes

Estudiante D:
  Exam3: Lunes
  Exam5: Martes
  Exam7: Miércoles

Tiempo: 0.0002 s
----------------------------------------

=== Beam Search ===
Estudiante A:
  Exam1: Lunes
  Exam2: Martes
  Exam3: Miércoles

Estudiante B:
  Exam1: Martes
  Exam4: Lunes
  Exam5: Miércoles

Estudiante C:
  Exam2: Lunes
  Exam6: Miércoles
  Exam7: Martes

Estudiante D:
  Exam3: Lunes
  Exam5: Martes
  Exam7: Miércoles

Tiempo: 0.0020 s
----------------------------------------

=== Local Search (Min-Conflicts) ===
Estudiante A:
  Exam1: Martes
  Exam2: Miércoles
  Exam3: Lunes

Estudiante B:
  Exam1: Miércoles
  Exam4: Martes
  Exam5: Lunes

Estudiante C:
  Exam2: Lunes
  Exam6: Miércoles
  Exam7: Martes

Estudiante D:
  Exam3: Martes
  Exam5: Miércoles
  Exam7: Lunes

Tiempo: 0.7671 s
-------